# Uni-Projekt: Large Language Models from Scratch
**Gruppe:** Phillip, Konstantin, Fabian

**Ziel:** Aufbau und Training eines eigenen LLM-Modells, mithilfe eines Datensatzes der Kochrezepte enthält.

# Datensatz einlesen
* Den CSV Datensatz einlesen
* Die ersten Zeilen ausgeben

In [37]:
import os
import pandas as pd
from sympy import root

root_dir = file_path = os.path.join("C:\\", "resources", "Uni", "LLMs", "LLMs-from-scratch")
file_path = file_path = os.path.join(root_dir, "datasets", "reduced_dataset_100k.csv")

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(df.info())
    print(df.head())   
else:    print(f"File '{file_path}' does not exist.")



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Unnamed: 0   100000 non-null  int64 
 1   title        100000 non-null  object
 2   ingredients  100000 non-null  object
 3   directions   100000 non-null  object
 4   link         100000 non-null  object
 5   source       100000 non-null  object
 6   NER          100000 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB
None
   Unnamed: 0                  title  \
0           0    No-Bake Nut Cookies   
1           1  Jewell Ball'S Chicken   
2           2            Creamy Corn   
3           3          Chicken Funny   
4           4   Reeses Cups(Candy)     

                                         ingredients  \
0  ["1 c. firmly packed brown sugar", "1/2 c. eva...   
1  ["1 small jar chipped beef, cut up", "4 boned ...   
2  ["2 (16 oz.) pkg. frozen corn", "1 (8 oz.) pkg

- Die CSV-Datei als einen langen String formatieren

In [38]:
def format_csv(row):
    title = str(row['title'])
    ingredients = str(row['ingredients']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    directions = str(row['directions']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    return f"Recepie: {title}\nIngredients:{ingredients}\nDirections:{directions}"

raw_text = "".join(df.apply(format_csv, axis=1))
print(f"Der gesamte Datensatz wurde in einen String umgewandelt.")
print(f"Gesamtanzahl der Zeichen: {len(raw_text)}")
print("\nVorschau der ersten 300 Zeichen:")
print(raw_text[:300])

#for test purposes
raw_text = raw_text[:100]
    

Der gesamte Datensatz wurde in einen String umgewandelt.
Gesamtanzahl der Zeichen: 46874254

Vorschau der ersten 300 Zeichen:
Recepie: No-Bake Nut Cookies
Ingredients:1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/2 tsp. vanilla, 1/2 c. broken nuts (pecans), 2 Tbsp. butter or margarine, 3 1/2 c. bite size shredded rice biscuits
Directions:In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and bu


- Text kürzen

# Tokenizer bauen


- Tokenizer auf mit einfachen Textbeispielen bauen und später auf den Datensatz anwenden
- Mithilfe von RegEx die Wörter trennen

In [39]:
import re

text = "This is a sample text: with numbers 123, special characters !@# and a few more.  ."
def split_text(text):
    tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    tokens = [token for token in tokens if token.strip()]
    return tokens


result = split_text(text)


print(result)

['This', 'is', 'a', 'sample', 'text', ':', 'with', 'numbers', '123', ',', 'special', 'characters', '!', '@#', 'and', 'a', 'few', 'more', '.', '.']


- Trennung der Wörter auf den Datensatz anwenden

In [40]:
split = split_text(raw_text)
print(f"Anzahl der Wörter: {len(split)}")
print(split[:20])

Anzahl der Wörter: 22
['Recepie', ':', 'No-Bake', 'Nut', 'Cookies', 'Ingredients', ':', '1', 'c', '.', 'firmly', 'packed', 'brown', 'sugar', ',', '1/2', 'c', '.', 'evaporated', 'milk']


&nbsp;
## 2.3 Converting tokens into token IDs

- Vokabular das alle Wort-Tokens im Datensatzenthält

In [41]:
all_words = sorted(set(split))
vocab_size = len(all_words)
print(vocab_size)

17


In [42]:
vocab = {token:integer for integer,token in enumerate(all_words)}

In [43]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

(',', 0)
('.', 1)
('1', 2)
('1/2', 3)
(':', 4)
('Cookies', 5)
('Ingredients', 6)
('No-Bake', 7)
('Nut', 8)
('Recepie', 9)
('brown', 10)
('c', 11)
('evaporated', 12)
('firmly', 13)
('milk', 14)
('packed', 15)
('sugar', 16)


In [44]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [45]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Recepie: No-Bake Nut Cookies Ingredients: 1 c. firmly packed brown sugar."
ids = tokenizer.encode(text)
print(ids)

[9, 4, 7, 8, 5, 6, 4, 2, 11, 1, 13, 15, 10, 16, 1]


In [46]:
tokenizer.decode(ids)

'Recepie : No-Bake Nut Cookies Ingredients : 1 c. firmly packed brown sugar.'

In [47]:
tokenizer.decode(tokenizer.encode(text))

'Recepie : No-Bake Nut Cookies Ingredients : 1 c. firmly packed brown sugar.'

# Add specials tokens
- unknown
- end of text
- GPT2 semantic

In [48]:
all_tokens = sorted(list(set(split)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [49]:
#adjusted tokenizer
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [50]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [51]:
tokenizer.encode(text)

[18, 0, 18, 18, 18, 18, 18, 17, 18, 18, 18, 18, 18, 18, 18, 1]

In [52]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|endoftext|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|>.'

# Byte Pair Encoding

Usage of a an already implemented byte pair encoding from open ai for further development of the LLM. The previous tokenizer was just for training purposes.

In [53]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.9.0


In [54]:
tokenizer = tiktoken.get_encoding("cl100k_base")

In [55]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[9906, 11, 656, 499, 1093, 15600, 30, 220, 100257, 763, 279, 7160, 32735, 7317, 2492, 1073, 1063, 16476, 17826, 13]


In [56]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


# Data sampling

for example checkout ch02

In [57]:
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

34


In [58]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.10.0


In [59]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [60]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [61]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[ 3041,   344, 21749,    25]]), tensor([[  344, 21749,    25,  1400]])]


In [62]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[  344, 21749,    25,  1400]]), tensor([[21749,    25,  1400,    12]])]


# Token embeddings

In [63]:
input_ids = torch.tensor([2, 3, 5, 1])

In [64]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
# --> das Embedding Layer ist basically einfach eine simple Matrix, welches als Look up dient.
# Es wird trotzdem als Layer definiert, damit es in einem Modell integriert werden kann und die Gewichte automatisch aktualisiert werden können. Obwohl es keine eigentliche Verarbeitung gibt.

In [65]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [66]:
# convert token id 2 to 3 dimensional vector
print(embedding_layer(torch.tensor([2])))

tensor([[ 1.2753, -0.2010, -0.1606]], grad_fn=<EmbeddingBackward0>)


In [67]:
 # embedding every token
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


# Encoding word positions to the embedding matrix

In [68]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [69]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [71]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[ 3041,   344, 21749,    25],
        [ 1400,    12,    33,   539],
        [11959, 45305,   198, 41222],
        [   25,    16,   269,    13],
        [14245, 11856,  7586,  7543],
        [   11,   352,    14,    17],
        [  269,    13, 28959,   515],
        [ 7545,    11,   352,    14]])

Inputs shape:
 torch.Size([8, 4])


In [73]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(token_embeddings)

torch.Size([8, 4, 256])


In [74]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# uncomment & execute the following line to see how the embedding layer weights look like
# print(pos_embedding_layer.weight)

In [75]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(pos_embeddings)

torch.Size([4, 256])


In [76]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(input_embeddings)

torch.Size([8, 4, 256])


### TODO: 
- byte pair tiktokenizer vergleichen und unterschiede erklären (gpt2 vs cl100k_base)
- ist es möglich mit dem eigenen Tokenizer bereits "eigene" Sätze zu bilden, welche so nicht in dem Datensatz vorkommen und nichts mit Rezepten an sich zu tun haben